# Apex Planet Task 4 — Advanced Analytics & Statistical Modeling
Run the cells from the project root after installing `requirements.txt`. The CSV files are synthetic practice data.

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sales = pd.read_csv(ROOT / 'data' / 'daily_sales.csv', parse_dates=['Date'])
customers = pd.read_csv(ROOT / 'data' / 'customer_churn.csv')
display(sales.head())
display(customers.head())

## 1. Time series: resampling and stationarity

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
daily = sales.set_index('Date')['Sales'].asfreq('D').interpolate()
weekly = daily.resample('W').sum()
monthly = daily.resample('MS').sum()
display(monthly.head())
print('ADF statistic and p-value:', adfuller(daily)[0:2])
monthly.plot(title='Monthly sales total', figsize=(10,4)); plt.tight_layout(); plt.show()

## 2. Customer segmentation: K-Means and PCA

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
features = ['TenureMonths','MonthlyCharges','TotalCharges','SupportCalls']
X = StandardScaler().fit_transform(customers[features])
scores = {k: silhouette_score(X, KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)) for k in range(2,7)}
print('Silhouette scores:', scores)
best_k = max(scores, key=scores.get)
customers['Segment'] = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(X)
coords = PCA(n_components=2, random_state=42).fit_transform(X)
plt.figure(figsize=(8,5)); plt.scatter(coords[:,0], coords[:,1], c=customers['Segment'], alpha=.65)
plt.title('Customer segments (PCA view)'); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.tight_layout(); plt.show()
display(customers.groupby('Segment')[features + ['Churn']].mean().round(2))

## 3. Predictive modeling: churn classification

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
target = 'Churn'
num = ['TenureMonths','MonthlyCharges','TotalCharges','SupportCalls']
cat = ['ContractType','InternetService']
X = customers[num + cat]; y = customers[target]
prep = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')),('scale', StandardScaler())]), num), ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
model = Pipeline([('prep', prep), ('model', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'))])
model.fit(X_train, y_train); pred = model.predict(X_test)
print(classification_report(y_test, pred, zero_division=0))
print('ROC-AUC:', round(roc_auc_score(y_test, model.predict_proba(X_test)[:,1]), 4))